# CIDS · 07 — S4: frequency adapter for SAM-S2b (⑤, no metadata)

SAM's own input crop → **3D Haar transform (8 directional bands)** → **learned band weights** → small CNN → residual added to SAM's image embedding (zero at start, so SAM = S2b at epoch 0). Only the adapter trains (~333k parameters); SAM-S2b stays fixed, so any change is the adapter's. Same box mix as S2b (clean / messy / negative + clicks).

Prints the learned **band weights** (which frequency bands the checker relies on) and compares **S2b vs S2b + adapter** on identical test boxes.

**Settings:** GPU T4 x2 · **Internet ON** · Persistence: Files.  
**Inputs:** cache · `cids-sam-emb` · `cids-checkpoints` (`sam_s2b_best.pt`).  
**Time:** ~1.5 h. **After the run:** download `s4_adapter/best.pt` → upload to `cids-checkpoints` as **`sam_s2b_adapter_best.pt`**.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
EMB_ROOT   = '/kaggle/input/datasets/kumarvemuri1/cids-sam-emb/SAM_embeddings_dataset'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
SAM_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'
CODE_DIR   = '/kaggle/working/cids'
EPOCHS = 40
OUT = '/kaggle/working/s4_adapter'
PATHS = [CACHE_ROOT, EMB_ROOT, SAM_CKPT]


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q monai medim einops appdirs
!pytest -q tests

In [ ]:
import os, torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
print('GPU:', torch.cuda.get_device_properties(0).name)
for p in PATHS:
    print('ok     ' if Path(p).exists() else 'MISSING', p)


## Smoke test — 16 train, 1 epoch, 8 val + 8 test (~5 min)
Check: the `epoch 0 (= S2b)` line appears, then one epoch line with `bands LLL=… HHH=…`, then a test table with `s2b` and `s2b_adapter`.

In [ ]:
!python scripts/s4_sam_adapter.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --sam_ckpt {SAM_CKPT} \
    --out /kaggle/working/s4_smoke --epochs 1 --val_every 1 --limit_train 16 --limit_eval 8
!rm -rf /kaggle/working/s4_smoke


## Full run (~1.5 h) — *Save Version → Save & Run All*

In [ ]:
!python scripts/s4_sam_adapter.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --sam_ckpt {SAM_CKPT} \
    --out {OUT} --epochs {EPOCHS}


In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/band_weights.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
!du -sh {OUT}
